In [ ]:
import numpy as np
import pandas as pd

# ========= 1) Excel 读列 =========
file_path = r"your_data.xlsx"
sheet_name = "DEA"
input_cols = ["in1","in2"]       # 输入指标列
output_cols = ["out1","out2"]    # 输出指标列

df = pd.read_excel(file_path, sheet_name=sheet_name)
X = df[input_cols].to_numpy(dtype=float)
Y = df[output_cols].to_numpy(dtype=float)

# ========= 2) 参数模板 =========
params = {
    "orientation": "input"  # str: 'input'/'output'（具体建模时使用）
}

# 注：DEA一般对每个DMU单独建线性规划，这里仅保留数据读取和参数框架
print("X shape=", X.shape, "Y shape=", Y.shape, "orientation=", params["orientation"])


# 数据包络分析

## 输入说明

- 数据文件：默认读取脚本同目录下的 `data.csv`，也可以在代码顶部把 `DATA_FILE` 改为 `.xlsx` 或绝对路径。
- 数据格式：一般要求“一行一个样本/时刻/方案，一列一个变量/指标”。具体列名需要在代码顶部的 `TODO` 参数区填写。
- 示例：若模型需要特征 `特征1、特征2` 和目标列 `y`，表格可整理为：

| 特征1 | 特征2 | y |
|---:|---:|---:|
| 1.2 | 3.4 | 8.1 |
| 2.0 | 2.8 | 9.0 |

## 输出说明

- 控制台会打印核心结果，例如模型参数、评价指标、最优解、排名或预测值。
- 默认结果保存到代码顶部 `OUTPUT_FILE` 指定的文件。
- 若模型包含图形分析，会额外输出图片文件，例如箱型图 `boxplot.png`。

## 原理通俗解释

数据包络分析 的核心思想是：先把实际问题抽象成可计算的数据结构，再用对应的数学规则寻找“预测值、分类结果、综合得分或最优方案”。代码中已经保留主要计算流程，比赛时重点是把题目数据整理成表格，并把 TODO 参数替换为题目含义一致的列名和约束。

## 适用场景

多投入、多产出的相对效率评价。

## 局限性

DMU 数量太少或指标选择不合理会造成区分度不足。

## 使用提示

- 运行前先检查缺失值、异常值和量纲；很多模型对数据尺度敏感。
- 所有 `TODO` 都应结合题目背景填写，不要直接使用示例列名。
- 建模论文中建议同时写明参数来源，例如权重来自 AHP/熵权法，预测步数来自题目要求。

In [ ]:
"""
数据包络分析

使用方法：
1. 按照下方 TODO 修改 DATA_FILE、列名、参数和输出文件名。
2. 将数据文件放在本脚本同目录，或把 DATA_FILE 改成绝对路径。
3. 运行：python "数据包络分析.py"
"""

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.optimize import linprog



DATA_FILE = "data.csv"  # TODO: 请填写[数据文件路径]，说明：CSV/Excel 均可；若使用 Excel，请在 load_data 中改为 read_excel。
OUTPUT_FILE = "model_output.csv"  # TODO: 请填写[输出文件名]，说明：保存模型结果，建议保留 .csv 或 .xlsx 后缀。
RANDOM_STATE = 42  # TODO: 请填写[随机种子]，说明：用于复现实验；整数即可。
DMU_COLUMN = "对象"  # TODO: 请填写[决策单元列名]，说明：每行一个评价对象。
INPUT_COLUMNS = ["投入1", "投入2"]  # TODO: 请填写[投入指标列名]，说明：越少越好。
OUTPUT_COLUMNS = ["产出1", "产出2"]  # TODO: 请填写[产出指标列名]，说明：越多越好。



REQUIRES_DATA = True  # 参数型模型可不提供数据文件；表格型模型必须提供数据。


def load_data() -> pd.DataFrame:
    """读取用户数据；竞赛时通常把 Excel/CSV 表格整理成一行一个样本。"""
    path = Path(DATA_FILE)
    if not path.exists():
        if not REQUIRES_DATA:
            return pd.DataFrame()
        raise FileNotFoundError(
            f"未找到数据文件 {DATA_FILE}。请先修改 DATA_FILE，或将数据放到脚本同目录。"
        )
    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    return pd.read_csv(path)


def run_model(data: pd.DataFrame) -> None:
    # CCR 输入导向 DEA：每个 DMU 单独求一次线性规划。
    inputs = data[INPUT_COLUMNS].to_numpy(dtype=float)
    outputs = data[OUTPUT_COLUMNS].to_numpy(dtype=float)
    n = len(data)
    scores = []
    for i in range(n):
        c = np.r_[np.zeros(n), 1.0]
        A_ub = []
        b_ub = []
        for k in range(inputs.shape[1]):
            A_ub.append(np.r_[inputs[:, k], -inputs[i, k]])
            b_ub.append(0)
        for r in range(outputs.shape[1]):
            A_ub.append(np.r_[-outputs[:, r], 0])
            b_ub.append(-outputs[i, r])
        bounds = [(0, None)] * n + [(0, None)]
        res = linprog(c, A_ub=np.array(A_ub), b_ub=np.array(b_ub), bounds=bounds, method="highs")
        scores.append(res.x[-1] if res.success else np.nan)
    result = data[[DMU_COLUMN]].copy()
    result["DEA效率"] = scores
    result.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
    print(result)


if __name__ == "__main__":
    df = load_data()
    run_model(df)
